# Phase 2 — Trip distribution and revenue optimization

Run the cells in order. The [English report](../reports/en/phase-2-trip-distribution-and-revenue.md)
defines the comparison and the scope of the optimization. Obtain the input CSV using the
[data instructions](../data/README.md).


# بخش اول: تحلیل ماتریس OD

### قسمت اول: ساخت ماتریس OD

In [ ]:
import pandas as pd

from pathlib import Path

# Jupyter may start the kernel in notebooks/ or at the repository root.
data_dir = Path("data") if Path("data").is_dir() else Path("../data")
data_file = data_dir / "rideshare_kaggle.csv"
if not data_file.is_file():
    raise FileNotFoundError(f"Place the Kaggle CSV at {data_file}. See data/README.md.")

phase2_columns = [
    "hour", "day", "month", "source", "destination", "cab_type", "price", "distance",
]
df = pd.read_csv(data_file, usecols=phase2_columns)

#ابتدا باید ببینیم چند زون و چه زون‌هایی داریم
zones = df['source'].unique()
print(zones)


In [ ]:
# ایجاد یک ماتریس خالی
OD = pd.DataFrame( index = zones, columns = zones)

# نام گذاری خانه گوشه بالا سمت چپ
OD.index.name = 'origin'
OD.columns.name = 'destination'


# پر کردن ماتریس
for origin in OD.index:
    for destination in OD.columns:
        count = len(df[(df['source'] == origin) & (df['destination'] == destination)])
        OD.loc[origin, destination] = count



In [ ]:
OD.style.set_table_styles([{'selector': 'th', 'props': [('font-size', '12px')]}])

In [ ]:
import numpy as np
# چک کردن متقارن بودن : اگر متقارن باشد با ترانهاده‌اش برابر است
print(OD.equals(OD.T))

#چک کردن بالانس بودن
OD_numeric = OD.astype(float)
row_sums = OD_numeric.sum(axis=1)  # مجموع هر origin
col_sums = OD_numeric.sum(axis=0)  # مجموع هر destination
# tolerance برای کوچک‌ترین اختلاف‌ها
tol = 0.1
balanced = np.allclose(row_sums.values, col_sums.values, atol=tol)
if balanced:
    print(' Balanced')
else:
    print(' Not balanced')



### قسمت دوم: برای تمام روزهای هفته

In [ ]:
# چک کردن این حدس که کل دیتا مربوط به ۲ ماه آخر سال است یا خیر
print(df['month'].unique())

In [ ]:
# دو تابع برای هر ماه تعریف می‌کنیم تا روز را برایمان پیدا کند

def what_day_december(column):
    num = int(column) % 7
    if num == 5:
        return 'Wednesday'
    elif num == 6:
        return 'Thursday'
    elif num == 0:
        return 'Friday'
    elif num == 1:
        return 'Saturday'
    elif num == 2:
        return 'Sunday'
    elif num == 3:
        return 'Monday'
    else:
        return 'Tuesday'

def what_day_november(column):
     num = int(column) % 7
     if num == 0:
        return 'Wednesday'
     elif num == 1:
        return 'Thursday'
     elif num == 2:
        return 'Friday'
     elif num == 3:
        return 'Saturday'
     elif num == 4:
        return 'Sunday'
     elif num == 5:
        return 'Monday'
     else:
        return 'Tuesday'



In [ ]:
# روزهای هفته
days = ['Wednesday', 'Thursday', 'Friday',
        'Saturday', 'Sunday', 'Monday', 'Tuesday']

# کپی گرفتن از دیتافریم اصلی
df = df.copy()

#ایجاد کردن ستونی که می‌گوید آن روز چه روزی است
df['weekday'] = np.where(
    df['month'] == 11,
    df['day'].map(what_day_november),
    df['day'].map(what_day_december)
)


grouped = (
    df
    .groupby(['weekday', 'source', 'destination'])
    .size()
)


# دیکشنری ای که در آن ماتریس‌ها قرار است ذخیره شوند
OD_dic = {}

for day in days:
    OD_dic[day] = (
        grouped
        .get(day, pd.Series(dtype=int))
        .unstack(fill_value=0)
        .reindex(index=zones, columns=zones, fill_value=0)
    )

    OD_dic[day].index.name = 'origin'
    OD_dic[day].columns.name = 'destination'

In [ ]:
for day, od in OD_dic.items():
    print('-' * 40)
    print(' ' * 14, day)
    print('-' * 40)

    #  قسمت دوم: محاسبه میانگین درایه های غیر صفر برای هر ماتریس
    non_zero = od.values[od.values != 0]
    avg_non_zero = non_zero.mean() if non_zero.size > 0 else 0
    print(f'\n Average: {avg_non_zero:.2f}')

    #قسمت سوم: چک کردن بالانس بودن یا نبودن هر ماتریس
    row_sums = od.sum(axis=1)  # مجموع هر origin
    col_sums = od.sum(axis=0)  # مجموع هر destination
    # tolerance برای کوچک‌ترین اختلاف‌ها
    tol = 0.1
    balanced = np.allclose(row_sums.values, col_sums.values, atol=tol)
    if balanced:
        print(' Balanced')
    else:
        print(' Not balanced')

    display(od)





In [ ]:
OD0 = OD.copy()

# Furness

In [ ]:
import numpy as np
import pandas as pd

def furness(T0, O, D, max_iter=1000, tol_percent=1.0, balance_D=True):
    """
    T0: DataFrame (OD پایه)
    O : Series (تولید هدف)
    D : Series (جذب هدف)
    """

    zones = list(T0.index)
    T = T0.to_numpy(dtype=float)

    O = O.loc[zones].to_numpy(dtype=float)
    D = D.loc[T0.columns].to_numpy(dtype=float)


    if balance_D:
        sumO, sumD = O.sum(), D.sum()
        if sumD == 0:
            raise ValueError("Sum(D)=0")
        D = D * (sumO / sumD)

    T_total = O.sum()

    for k in range(1, max_iter + 1):
        # 1) بالانس سطری
        row_sum = T.sum(axis=1)
        if np.any(row_sum == 0):
            raise ValueError("Zero row sum -> cannot balance rows")
        r = O / row_sum
        T1 = (r[:, None]) * T


        col_sum = T1.sum(axis=0)
        if np.any(col_sum == 0):
            raise ValueError("Zero col sum -> cannot balance cols")
        c = D / col_sum
        T_new = T1 * (c[None, :])

        # 3) خطا (جمع انحرافات / کل سفر) * 100
        row_new = T_new.sum(axis=1)
        col_new = T_new.sum(axis=0)
        err = (np.abs(O - row_new).sum() + np.abs(D - col_new).sum()) / T_total * 100

        T = T_new
        if err < tol_percent:
            break

    T_df = pd.DataFrame(T, index=zones, columns=T0.columns)
    return T_df, err, k

future = pd.DataFrame({
    "zone": [
        "Financial District","Theatre District","Back Bay","Haymarket Square",
        "Boston University","Fenway","North End","Northeastern University",
        "South Station","West End","Beacon Hill","North Station"
    ],
    "O_future": [56850,57820,57799,57836,57714,57750,57963,55756,58750,57560,57483,56228],
    "D_future": [59700,57900,58600,56840,59500,56750,58750,56950,58700,56600,58200,56200]
}).set_index("zone")

# خیلی مهم: ترتیب zone ها باید با index/columns ماتریس OD0 یکی باشد
O_target = future.loc[OD0.index, "O_future"]
D_target = future.loc[OD0.columns, "D_future"]

T_furness, err, iters = furness(OD0, O_target, D_target, max_iter=1000, tol_percent=1.0)
print("iters:", iters, "err%:", err)
furness_iterations, furness_error = iters, err


# Detroit

In [ ]:
import numpy as np
import pandas as pd

def detroit_using_growth_factors(
    T0, O, D,
    max_iter=1000,
    g_low=0.99, g_high=1.01,
    balance_D=True
):
    zones = list(T0.index)
    cols  = list(T0.columns)

    T = T0.to_numpy(dtype=float)
    O = O.loc[zones].to_numpy(dtype=float)
    D = D.loc[cols].to_numpy(dtype=float)

    if balance_D:
        sumO, sumD = O.sum(), D.sum()
        if sumD == 0:
            raise ValueError("Sum(D)=0")
        D = D * (sumO / sumD)

    sumO = O.sum()

    for k in range(1, max_iter + 1):

        row_sum = T.sum(axis=1)
        col_sum = T.sum(axis=0)

        if np.any(row_sum == 0) or np.any(col_sum == 0):
            raise ValueError("Some row/column totals are zero -> cannot compute growth factors.")


        F_i = O / row_sum
        F_j = D / col_sum


        T_total_current = T.sum()
        if T_total_current == 0:
            raise ValueError("Total trips in T became zero.")
        F_ave = sumO / T_total_current


        T_new = (F_i[:, None] * F_j[None, :] / F_ave) * T

    
        row_new = T_new.sum(axis=1)
        col_new = T_new.sum(axis=0)

        g_row = O / row_new
        g_col = D / col_new

        T = T_new

        if (g_row.min() >= g_low and g_row.max() <= g_high and
            g_col.min() >= g_low and g_col.max() <= g_high):
            break

    T_df = pd.DataFrame(T, index=zones, columns=cols)
    return T_df, k, g_row.min(), g_row.max(), g_col.min(), g_col.max()

T_detroit2, iters, gr_min, gr_max, gc_min, gc_max = detroit_using_growth_factors(
    OD0, O_target, D_target,
    max_iter=1000, g_low=0.99, g_high=1.01, balance_D=True
)

print("iters:", iters)
print("row growth min/max:", gr_min, gr_max)
print("col growth min/max:", gc_min, gc_max)
detroit_iterations = iters


# Fratar

In [ ]:
import numpy as np
import pandas as pd

def fratar(T0, O, D, max_iter=1000, tol_percent=1.0, balance_D=True):

    zones = list(T0.index)
    cols  = list(T0.columns)

    T = T0.to_numpy(dtype=float)
    O = O.loc[zones].to_numpy(dtype=float)
    D = D.loc[cols].to_numpy(dtype=float)

    if balance_D:
        sumO, sumD = O.sum(), D.sum()
        if sumD == 0:
            raise ValueError("Sum(D)=0")
        D = D * (sumO / sumD)

    T_total = O.sum()

    for k in range(1, max_iter + 1):
        t_i = T.sum(axis=1)  
        t_j = T.sum(axis=0)  

        if np.any(t_i == 0) or np.any(t_j == 0):
            raise ValueError("Some row/column sums are zero -> cannot compute growth factors.")

        f_i = O / t_i
        f_j = D / t_j

        weighted = T * f_j              
        denom = weighted.sum(axis=1)   

        if np.any(denom == 0):
            raise ValueError("Denominator became zero in some row during update.")

        T_new = (t_i * f_i)[:, None] * (weighted / denom[:, None])

        row_new = T_new.sum(axis=1)
        col_new = T_new.sum(axis=0)

        err = (np.abs(O - row_new).sum() + np.abs(D - col_new).sum()) / T_total * 100

        T = T_new
        if err < tol_percent:
            break

    T_df = pd.DataFrame(T, index=zones, columns=cols)
    return T_df, err, k
T_fratar, err, iters = fratar(OD0, O_target, D_target, max_iter=1000, tol_percent=1.0)
print("iters:", iters, "err%:", err)
fratar_iterations, fratar_error = iters, err


# C Matrix

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(data_file, usecols=phase2_columns)

df["price"] = pd.to_numeric(df["price"], errors="coerce")
cost = (
    df.groupby(["source", "destination"], as_index=False)["price"]
      .mean()
)


C = cost.pivot(index="source", columns="destination", values="price")

# 6) NaN ها = بی‌نهایت
C = C.fillna(np.inf)

#قطر اصلی بی نهایت میشود زیرا در ماتریس اولیه سفر در داخل زون نداریم و قطر صفر است
np.fill_diagonal(C.values, np.inf)

C.head()


In [ ]:
C = C.reindex(index=OD0.index, columns=OD0.columns, fill_value=np.inf)

print("C shape:", C.shape, " OD0 shape:", OD0.shape)
print("Any inf in C?", np.isinf(C.values).any())


In [ ]:
row_all_inf = np.isinf(C.values).all(axis=1)
col_all_inf = np.isinf(C.values).all(axis=0)

print("rows all inf:", list(C.index[row_all_inf]))
print("cols all inf:", list(C.columns[col_all_inf]))


# Gravity exponential

In [ ]:
import numpy as np
import pandas as pd

def gravity_exponential(C, O, D, alpha=0.05, beta=0.2, max_iter=1000, tol_percent=1.0, balance_D=True):

    zones = list(C.index)
    cols  = list(C.columns)

    # Align
    O = O.loc[zones].astype(float).to_numpy()
    D = D.loc[cols].astype(float).to_numpy()
    Cnp = C.to_numpy(dtype=float)

    if balance_D:
        sumO, sumD = O.sum(), D.sum()
        if sumD == 0:
            raise ValueError("Sum(D)=0")
        D = D * (sumO / sumD)

    # friction factor
    F = alpha * np.exp(-beta * Cnp)

    # Initialize B (or A)
    B = np.ones(len(cols), dtype=float)

    T_total = O.sum()

    for k in range(1, max_iter + 1):
        # Update A using current B
        denom_A = (F * (B[None, :] * D[None, :])).sum(axis=1)  # sum_j Fij * Bj * Dj
        if np.any(denom_A == 0):
            raise ValueError("Zero denom_A: check F (maybe too small) or D/B.")
        A = 1.0 / denom_A

        # Update B using current A
        denom_B = (F * (A[:, None] * O[:, None])).sum(axis=0)  # sum_i Fij * Ai * Oi
        if np.any(denom_B == 0):
            raise ValueError("Zero denom_B: check F (maybe too small) or O/A.")
        B_new = 1.0 / denom_B

        # Build T
        T = (A[:, None] * O[:, None]) * (B_new[None, :] * D[None, :]) * F

        # Check error (percent)
        row_new = T.sum(axis=1)
        col_new = T.sum(axis=0)
        err = (np.abs(O - row_new).sum() + np.abs(D - col_new).sum()) / T_total * 100

        B = B_new
        if err < tol_percent:
            break

    T_df = pd.DataFrame(T, index=zones, columns=cols)
    return T_df, err, k
T_grav, err, iters = gravity_exponential(
    C,
    O_target,
    D_target,
    alpha=0.05,
    beta=0.2,
    max_iter=1000,
    tol_percent=1.0
)

print("iters:", iters, "err%:", err)
exponential_iterations, exponential_error = iters, err


In [ ]:
diag_share = np.diag(T_grav.values).sum() / T_grav.values.sum()
print("Diagonal share:", diag_share)


# gravity power Function

In [ ]:
import numpy as np
import pandas as pd

def gravity_power(C, O, D, alpha=1.0, beta=1.5, max_iter=1000, tol_percent=1.0, balance_D=True):
    zones = list(C.index)
    cols  = list(C.columns)

    O = O.loc[zones].astype(float).to_numpy()
    D = D.loc[cols].astype(float).to_numpy()
    Cnp = C.to_numpy(dtype=float)

    if balance_D:
        D = D * (O.sum() / D.sum())

    # جلوگیری از تقسیم بر صفر
    F = alpha * np.where(np.isfinite(Cnp), Cnp ** (-beta), 0.0)

    B = np.ones(len(cols))
    T_total = O.sum()

    for k in range(1, max_iter + 1):
        denom_A = (F * (B[None, :] * D[None, :])).sum(axis=1)
        A = 1.0 / denom_A

        denom_B = (F * (A[:, None] * O[:, None])).sum(axis=0)
        B_new = 1.0 / denom_B

        T = (A[:, None] * O[:, None]) * (B_new[None, :] * D[None, :]) * F

        err = (
            np.abs(T.sum(axis=1) - O).sum() +
            np.abs(T.sum(axis=0) - D).sum()
        ) / T_total * 100

        B = B_new
        if err < tol_percent:
            break

    
    return pd.DataFrame(T, index=zones, columns=cols), err, k
T_grav_power, err_p, iters_p = gravity_power(
    C,
    O_target,
    D_target,
    alpha=1.0,
    beta=1.5,
    max_iter=1000,
    tol_percent=1.0
)

print("iters:", iters_p, "err%:", err_p)
power_iterations, power_error = iters_p, err_p


In [ ]:
df.columns

## Part 3 — Daily revenue optimization

This reproduces the course model. The distance budget is 20 in the CSV's
native distance units. The submitted code labels those units as kilometers
and applies no conversion. Decision variables represent continuous trip volumes.


In [ ]:

import pandas as pd
import numpy as np
import math
import warnings
try:
    import pulp
except Exception:
    raise ImportError("Please install pulp first: pip install pulp")

# -------------- Step 1: infer weekday from month/day --------------

def infer_weekday_name(row):
    m = int(row['month'])
    d = int(row['day'])
    if m == 12: return what_day_december(d)
    if m == 11: return what_day_november(d)
    return None

df = df.copy()
df['weekday_name'] = df.apply(infer_weekday_name, axis=1)
weekday_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
df['weekday'] = df['weekday_name'].map(weekday_map)
df['date_id'] = df['month'].astype(int).astype(str).str.zfill(2) + "-" + df['day'].astype(int).astype(str).str.zfill(2)

# -------------- Step 2: slots (4 slots, 6 hours each) --------------
df['slot'] = (df['hour'] // 6).astype(int)
df['slot'] = df['slot'].clip(0,3)

# -------------- Step 3: restrict to weekdays (Mon-Fri) --------------
df = df[df['weekday'].isin([0,1,2,3,4])].copy()
if df.empty:
    raise ValueError("After filtering weekdays the dataframe is empty. Check date inference or data.")

# -------------- Step 4: compute price-per-km and filter invalid distances --------------
df = df[df['distance'] > 0].copy()
if df.empty:
    raise ValueError("No rows have positive distance.")

df['price_per_km'] = df['price'] / df['distance']

# -------------- Step 5: compute daily aggregates --------------
# trips per day × OD × slot
trips_day = df.groupby(['date_id','source','destination','slot']).size().rename('trips').reset_index()

# avg price_per_km per day × OD × slot × company
pricepk = df.groupby(['date_id','source','destination','slot','cab_type'])['price_per_km'] \
            .mean().reset_index().rename(columns={'price_per_km':'avg_price_per_km'})

# base price-per-km per day × OD × slot (avg across companies)
base_price_day = pricepk.groupby(['date_id','source','destination','slot'])['avg_price_per_km'] \
                        .mean().reset_index().rename(columns={'avg_price_per_km':'base_price_per_km'})

# avg distance per OD (across all days)
dist_od = df.groupby(['source','destination'])['distance'].mean().reset_index().rename(columns={'distance':'avg_distance'})

# merge params: keep only combinations that have observed trips (trips_day)
params = trips_day.merge(dist_od, on=['source','destination'], how='left') \
                  .merge(base_price_day, on=['date_id','source','destination','slot'], how='left')

# fill missing price_per_km with global mean if necessary, then compute price per trip
global_ppk = params['base_price_per_km'].mean()
if np.isnan(global_ppk):
    # fallback if no price info at all
    global_ppk = df['price_per_km'].mean() if 'price_per_km' in df.columns else 1.5
params['base_price_per_km'] = params['base_price_per_km'].fillna(global_ppk)
params['avg_distance'] = params['avg_distance'].fillna(df['distance'].mean())
params['base_price_per_trip'] = params['base_price_per_km'] * params['avg_distance']

# remove self-trips (origin == destination)
params = params[params['source'] != params['destination']].copy()
if params.empty:
    raise ValueError("After removing self-trips there are no OD pairs left to optimize.")

# ensure types
params['slot'] = params['slot'].astype(int)
params['trips'] = params['trips'].astype(float)
params['avg_distance'] = params['avg_distance'].astype(float)
params['base_price_per_trip'] = params['base_price_per_trip'].astype(float)

# -------------- Step 6: LP per day (robust) --------------
results_all = []
days = sorted(params['date_id'].unique())

for day in days:
    sub = params[params['date_id'] == day].copy()
    if sub.empty:
        continue

    # Build dictionaries using the same rows (guarantees matching keys)
    q = {}   # demand per (i,j,t)
    p = {}   # price per trip per (i,j,t)
    d = {}   # distance per (i,j)

    for r in sub.itertuples(index=False):
        i = r.source; j = r.destination; t = int(r.slot)
        q[(i,j,t)] = float(r.trips)           # observed trips in that day-slot
        p[(i,j,t)] = float(r.base_price_per_trip)
        d[(i,j)] = float(r.avg_distance)

    # If dictionaries are empty skip
    if not q:
        continue

    # build LP with variables ONLY for keys present in q (and requiring p,d present)
    prob = pulp.LpProblem(f"RevenueMax_{day}", pulp.LpMaximize)
    x_vars = {}
    for (i,j,t), qval in q.items():
        if (i,j) not in d or (i,j,t) not in p:
            # skip any incomplete record (shouldn't happen because we built from same rows)
            continue
        # continuous variables, 0 <= x <= qval
        x_vars[(i,j,t)] = pulp.LpVariable(f"x_{i}_{j}_{t}_{day}", lowBound=0, upBound=qval, cat='Continuous')

    if not x_vars:
        continue

    # objective and distance constraint
    prob += pulp.lpSum(p[(i,j,t)] * x_vars[(i,j,t)] for (i,j,t) in x_vars), "TotalRevenue"
    prob += pulp.lpSum(d[(i,j)] * x_vars[(i,j,t)] for (i,j,t) in x_vars) <= 20.0, "KilometerCap20"

    # solve quietly
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    # collect solutions and daily totals
    day_total_rev = 0.0
    day_total_km  = 0.0
    day_total_trips = 0.0
    status = pulp.LpStatus[prob.status] if hasattr(pulp,'LpStatus') else str(prob.status)

    for (i,j,t), var in x_vars.items():
        val = var.value()
        if val is None or val <= 1e-9:
            continue
        val = float(val)
        price = p.get((i,j,t), 0.0)
        dist  = d.get((i,j), 0.0)
        rev = price * val
        km  = dist * val

        results_all.append({
            'date_id': day,
            'origin': i,
            'destination': j,
            'slot': int(t),
            'served_trips': val,
            'distance_km': dist,
            'price_per_trip': price,
            'revenue': rev,
            'LP_status': status
        })
        day_total_rev += rev
        day_total_km += km
        day_total_trips += val

    # attach a day-level summary row in results_all as well (optional for convenience)
    results_all.append({
        'date_id': day,
        'origin': '__DAY_SUMMARY__',
        'destination': '',
        'slot': -1,
        'served_trips': day_total_trips,
        'distance_km': day_total_km,
        'price_per_trip': None,
        'revenue': day_total_rev,
        'LP_status': status
    })

# -------------- Step 7: results table & summary --------------
results_df = pd.DataFrame(results_all)
if results_df.empty:
    print("⚠️ No feasible positive solution found. Check params and input data.")
else:
    # separate summaries
    summary_rows = results_df[results_df['origin'] == '__DAY_SUMMARY__'][['date_id','served_trips','distance_km','revenue','LP_status']].copy()
    summary_rows = summary_rows.rename(columns={
        'served_trips':'total_served_trips',
        'distance_km':'total_km_used',
        'revenue':'total_revenue'
    }).set_index('date_id')

    print("=== Daily optimization summary ===")
    display(summary_rows)

    # detailed selected trips (exclude the summary rows)
    detail_df = results_df[results_df['origin'] != '__DAY_SUMMARY__'].copy()
    if not detail_df.empty:
        detail_df = detail_df.sort_values(['date_id','revenue'], ascending=[True,False]).reset_index(drop=True)
        print("\n=== Top selected OD-slot rows ===")
        display(detail_df.head(80))
    else:
        print("No individual OD-slot chosen by the solver (only day summaries exist).")

# -------------- Legend for slots --------------
print("\nSlot legend: 0 = 00:00–05:59, 1 = 06:00–11:59, 2 = 12:00–17:59, 3 = 18:00–23:59")


## Export results

In [ ]:
# Export the portfolio tables from the current run.
output_dir = data_dir.parent / "output"
output_dir.mkdir(exist_ok=True)
OD0.astype(int).to_csv(output_dir / "observed-od-matrix.csv")

weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_rows = []
for day in weekdays:
    matrix = OD_dic[day]
    values = matrix.to_numpy()
    nonzero = values[values > 0]
    weekday_rows.append({
        "weekday": day,
        "records": int(values.sum()),
        "nonzero_pairs": len(nonzero),
        "mean_records_per_nonzero_pair": nonzero.mean(),
        "balanced": bool(np.array_equal(values.sum(axis=0), values.sum(axis=1))),
    })
pd.DataFrame(weekday_rows).to_csv(output_dir / "weekday-od-summary.csv", index=False, float_format="%.6f")

distribution = [
    ("Furness", furness_iterations, furness_error),
    ("Detroit", detroit_iterations, np.nan),
    ("Fratar", fratar_iterations, fratar_error),
    ("Exponential gravity", exponential_iterations, exponential_error),
    ("Power gravity", power_iterations, power_error),
]
pd.DataFrame(distribution, columns=["method", "iterations", "aggregate_error_pct"]).to_csv(
    output_dir / "trip-distribution-convergence.csv", index=False, float_format="%.6f"
)

summary_rows.rename(columns={"total_km_used": "total_distance_raw_units"}).to_csv(
    output_dir / "daily-revenue.csv", float_format="%.6f"
)
detail_df.rename(columns={"distance_km": "distance_per_trip_raw_units"}).to_csv(
    output_dir / "revenue-allocation.csv", index=False, float_format="%.6f"
)
